# Notebook 01: Data Exploration

**Goal:** Understand what financial data yfinance gives us for Nifty 50 stocks.

Run this cell by cell. Read every output. This is how you learn what's in your data pipeline before writing production code.

In [ ]:
# Install if needed
# !pip install yfinance pandas matplotlib seaborn

import sys
sys.path.append('..')  # so we can import from src/

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded OK')

## 1. Fetch a single stock and explore all available data

In [ ]:
TICKER = 'RELIANCE.NS'  # Change this to any Nifty 50 ticker
stock = yf.Ticker(TICKER)

info = stock.info
print(f'Available keys in .info: {len(info)}')
print('\nSample keys:')
for k in list(info.keys())[:20]:
    print(f'  {k}: {info[k]}')

In [ ]:
# Key financial ratios
ratios = {
    'P/E (Trailing)': info.get('trailingPE'),
    'P/E (Forward)': info.get('forwardPE'),
    'P/B Ratio': info.get('priceToBook'),
    'EV/EBITDA': info.get('enterpriseToEbitda'),
    'Debt/Equity': info.get('debtToEquity'),
    'Current Ratio': info.get('currentRatio'),
    'ROE': f"{info.get('returnOnEquity', 0)*100:.1f}%" if info.get('returnOnEquity') else 'N/A',
    'Net Margin': f"{info.get('profitMargins', 0)*100:.1f}%" if info.get('profitMargins') else 'N/A',
    'Beta': info.get('beta'),
    'Market Cap (Cr)': f"₹{info.get('marketCap', 0)/1e7:,.0f} Cr" if info.get('marketCap') else 'N/A',
}

print(f'--- Key Ratios: {TICKER} ---')
for k, v in ratios.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# Income statement - last 4 years
print('--- Income Statement (Annual) ---')
financials = stock.financials
if not financials.empty:
    display(financials.head(6))
else:
    print('No data available')

In [ ]:
# Cash flow statement
print('--- Cash Flow (Annual) ---')
cf = stock.cashflow
if not cf.empty:
    # Show most important rows
    important_rows = ['Operating Cash Flow', 'Free Cash Flow', 'Capital Expenditure', 'Net Income']
    available = [r for r in important_rows if r in cf.index]
    if available:
        display(cf.loc[available])
    else:
        display(cf.head(5))

In [ ]:
# Price history + chart
hist = stock.history(period='5y', interval='1d')
print(f'Price history rows: {len(hist)}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(hist.index, hist['Close'], color='steelblue', linewidth=1)
ax1.set_title(f'{TICKER} — 5-Year Price Chart', fontsize=14)
ax1.set_ylabel('Price (INR)')
ax1.grid(True, alpha=0.3)

ax2.bar(hist.index, hist['Volume'], color='gray', alpha=0.6, width=2)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Compare multiple Nifty 50 stocks

In [ ]:
# Compare PE ratios across a sector
IT_STOCKS = ['INFY.NS', 'TCS.NS', 'WIPRO.NS', 'HCLTECH.NS', 'TECHM.NS']

comparison = []
for t in IT_STOCKS:
    s = yf.Ticker(t)
    i = s.info
    comparison.append({
        'Ticker': t.replace('.NS', ''),
        'PE Ratio': round(i.get('trailingPE', 0), 1),
        'ROE (%)': round((i.get('returnOnEquity') or 0) * 100, 1),
        'Net Margin (%)': round((i.get('profitMargins') or 0) * 100, 1),
        'Debt/Equity': round(i.get('debtToEquity') or 0, 2),
        'Market Cap (Cr)': round((i.get('marketCap') or 0) / 1e7, 0),
    })

df = pd.DataFrame(comparison).set_index('Ticker')
print('--- IT Sector Comparison ---')
display(df)

In [ ]:
# Visualize PE ratios
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

df['PE Ratio'].plot(kind='bar', ax=axes[0], color='steelblue', title='P/E Ratio')
df['ROE (%)'].plot(kind='bar', ax=axes[1], color='green', title='Return on Equity (%)')
df['Net Margin (%)'].plot(kind='bar', ax=axes[2], color='orange', title='Net Profit Margin (%)')

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('IT Sector — Key Metrics Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Key Observations

Write your observations here after running the notebook:

1. Which stock has the highest PE? Is it justified by its growth rate?
2. Which stock has the best ROE?
3. Any red flags (high debt, negative margins)?

This thinking process — before you even open Claude — is how institutional analysts work.